# IsoCourt — Conv3D+Pose (R(2+1)D) Training on FineBadminton-20K

Registry category: **`conv3d_pose`** | Script: `train_conv3d.py` | Checkpoint: `badminton_model_conv3d_pose.pth`

**Runtime → Change runtime type → GPU (A100 recommended, T4 works)**

In [ ]:
import torch, os
assert torch.cuda.is_available(), 'No GPU — change runtime to GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/IsoCourt/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

## 2 — Clone repo & install dependencies

In [ ]:
REPO = 'https://github.com/Navneethd8/IsoCourt.git'
BRANCH = 'main'

if not os.path.isdir('/content/IsoCourt'):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b {BRANCH} {REPO} /content/IsoCourt
else:
    !cd /content/IsoCourt && git pull --ff-only

%cd /content/IsoCourt

In [ ]:
!pip install -q opencv-python-headless mediapipe mlflow tqdm timm transformers safetensors huggingface_hub

## 3 — Download FineBadminton-20K & prepare data

In [ ]:
DATA_DIR = '/content/IsoCourt/backend/data/FineBadminton-20K'
MERGED_JSON = '/content/IsoCourt/backend/data/transformed_combined_rounds_output_en_evals_translated.json'

if not os.path.isfile(MERGED_JSON):
    !python backend/pipelines/vlm/common/prepare_finebadminton_20k.py \
        --local-dir {DATA_DIR} --max-workers 8
else:
    print(f'Merged JSON exists: {MERGED_JSON}')

## 4 — Train Conv3D+Pose

In [ ]:
EPOCHS = 60
SAVE_PATH = os.path.join(DRIVE_CKPT, 'badminton_model_conv3d_pose.pth')
POSE_CACHE = os.path.join(DRIVE_CKPT, 'pose_cache_mediapipe.pt')
USE_POSE = True

import sys
sys.path.insert(0, '/content/IsoCourt/backend')

from pipelines.training.train_conv3d import train_conv3d

train_conv3d(
    data_root='/content/IsoCourt/backend/data',
    list_file=MERGED_JSON,
    epochs=EPOCHS,
    batch_size=4,
    lr=1e-4,
    device='cuda',
    save_path=SAVE_PATH,
    pose_cache_path=POSE_CACHE,
    video_backbone='r2plus1d_18',
    spatial_size=112,
    pretrained=True,
    freeze_3d=True,
    unfreeze_layer4=True,
    lr_mult=5.0,
    backbone_lr_mult=0.1,
    weight_decay=1e-2,
    label_smoothing=0.1,
    scheduler_t0=10,
    scheduler_t_mult=2,
    accumulation_steps=4,
    stroke_loss_weight=2.0,
    aug_strength='strong',
    use_pose=USE_POSE,
)

In [ ]:
print('Training complete.')
!ls -lh {DRIVE_CKPT}/*conv3d* 2>/dev/null || echo 'No checkpoints found'